### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover 
- Documents
- Vector stores
- Retrievers


### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [ ]:
from langchain_core.documents import Document

## Create a list of Document objects with page content and metadata
documents = [
    Document(
        page_content = "Dogs are great companions, known for their loyalty and friendliness.",
        metadata = {"source": "mammal-pets-doc"},
    ),
    Document(
        page_content = "Cats are independent pets that often enjoy their own space.",
        metadata = {"source": "mammal-pets-doc"},
    ),
    Document(
        page_content = "Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata = {"source": "fish-pets-doc"},
    ),
    Document(
        page_content = "Parrots are intelligent birds capable of mimicking human speech.",
        metadata = {"source": "bird-pets-doc"},
    ),
    Document(
        page_content = "Rabbits are social animals that need plenty of space to hop around.",
        metadata = {"source": "mammal-pets-doc"},
    ),
]

documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [12]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key = groq_api_key, model = "llama-3.1-8b-instant")
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

print(llm)
print(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2575.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x000001AD68983E80> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AD68982E90> model_name='llama-3.1-8b-instant' model_kwargs={} groq_api_key=SecretStr('**********')
model_name='all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [13]:
## VectorStores
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding = embeddings)
print(vectorstore)

In [14]:
## Similarity Search on the vector store w.r.t cat
vectorstore.similarity_search("cat")

[Document(id='0c6abe36-cc02-423b-948d-556edaf086fb', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='4b4569ef-ca52-4567-9973-f5896c0132b4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='27d02cf2-c4a6-4002-839a-a5f1e764d4c4', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='88401f10-4d66-45d0-b1d5-58eba5eefc9c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [15]:
## Async query using asimilarity_search on the vector store w.r.t cat
await vectorstore.asimilarity_search("cat")

[Document(id='0c6abe36-cc02-423b-948d-556edaf086fb', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='4b4569ef-ca52-4567-9973-f5896c0132b4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='27d02cf2-c4a6-4002-839a-a5f1e764d4c4', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='88401f10-4d66-45d0-b1d5-58eba5eefc9c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [16]:
## Similarity Search with score on the vector store w.r.t cat
vectorstore.similarity_search_with_score("cat")

[(Document(id='0c6abe36-cc02-423b-948d-556edaf086fb', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058006286621),
 (Document(id='4b4569ef-ca52-4567-9973-f5896c0132b4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058006286621),
 (Document(id='27d02cf2-c4a6-4002-839a-a5f1e764d4c4', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='88401f10-4d66-45d0-b1d5-58eba5eefc9c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [17]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

## RunnableLambda for batch similarity search, binding k = 1 for top result
retriever = RunnableLambda(vectorstore.similarity_search).bind(k = 1)
retriever.batch(["cat", "dog"])

[[Document(id='0c6abe36-cc02-423b-948d-556edaf086fb', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='88401f10-4d66-45d0-b1d5-58eba5eefc9c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [18]:
## as_retriever to create a retriever object from the vectorstore
## binding search_type and search_kwargs
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 1}
)

## Batch retrieval using the retriever object
retriever.batch(["cat", "dog"])

[[Document(id='0c6abe36-cc02-423b-948d-556edaf086fb', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='88401f10-4d66-45d0-b1d5-58eba5eefc9c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [ ]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.
{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])
rag_chain = {
    "context" : retriever, "question" : RunnablePassthrough()
} | prompt | llm

response = rag_chain.invoke("tell me about dogs")
print(response.content)

Dogs are great companions, known for their loyalty and friendliness.
